## Импорты

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import talib

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from catboost import CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import bt

## Исходные данные

In [35]:
# Данные по индексу
# https://www.moex.com/ru/index/MOEXBC/archive?from=2009-04-24&till=2024-11-25&sort=TRADEDATE&order=desc

# Курс доллара
# https://investfunds.ru/indexes/39/

# !wget 'https://drive.google.com/uc?export=download&id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD' -O 'moexbc.csv'
# !wget 'https://drive.google.com/uc?export=download&id=1UYbOHL95Xe0MOEY7VOaTn17sto8MWl3B' -O 'usd_rub-(банк-россии).xlsx'

In [ ]:
# Индекс
db = pd.read_csv('moexbc.csv', encoding = 'windows-1251', sep = ';')
db = db.reset_index()
clns = list(db.loc[0, :])
db = db.iloc[1:, :].reset_index(drop=True)
db.columns = clns
db = db[['TRADEDATE','CLOSE','OPEN','HIGH','LOW','VALUE']]
db.columns = ['date', 'close','open', 'high', 'low', 'volume']
for c in ['close','open', 'high', 'low', 'volume']:
    db[c] = db[c].apply(lambda x: float(str(x).replace(',', '.')))
db['date'] = db['date'].apply(lambda x: str(x)[:10])

# Курс валюты
s0 = 'usd_rub-(банк-россии).xlsx'
d0 = pd.read_excel(s0)
d0.columns = ['date', 'USD_RUB']
d0['date'] = d0['date'].apply(lambda x: str(x)[:10])
d0['date'] = d0['date'].apply(lambda x: x[8:10]+'.'+x[5:7]+'.'+x[:4])
d0 = d0[['date', 'USD_RUB']]

# Собираем вместе
db = db.merge(d0, on='date', how='left')
db['close_RUR'] = db['close']
db['close'] = db['close']/db['USD_RUB']


# Обработка даты
c= 'date'
db[c] = db[c].apply(lambda x: x.split('.')[2]+'-'+x.split('.')[1]+'-'+x.split('.')[0])
db[c] = db[c].apply(lambda x: str(x))

c= 'date'
db[c] = db[c].apply(lambda x: str(x))
db = db.interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')  #type: ignore
db[c] = db[c].apply(pd.to_datetime, errors='coerce')
db = db.sort_values(c, ascending=True).reset_index(drop=True)

# Перевод всех значений в числовой формат
for c in [x for x in db.columns if x != 'date']:
    db[c] = db[c].apply(pd.to_numeric, errors='coerce')

print(db.shape)

db.head()

(3899, 8)


,date,close,open,high,low,volume,USD_RUB,close_RUR
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70


## Фильтруем данные
классический метод IQR-фильтрации (InterQuartile Range)

В scikit-learn нет готового трансформера “удалить выбросы по IQR”, поэтому напишем кастомный класс-трансформер и упакуем его в sklearn-совместимый трансформер (через BaseEstimator, TransformerMixin), чтобы встроить в Pipeline.

In [7]:
class IQRCarryForwardOutlierRemover(BaseEstimator, TransformerMixin):
    """
    Заменяет выбросы (по правилу IQR) на предыдущее корректное значение.
    - Выбросы: x < Q1 - factor*IQR или x > Q3 + factor*IQR
    - По умолчанию обрабатывает все числовые столбцы, кроме date-колонки.
    - Сохраняет DataFrame и порядок столбцов.
    """
    def __init__(self, factor=1.5, date_col='date', columns=None, sort_by_date=True):
        self.factor = float(factor)
        self.date_col = date_col
        self.columns = columns  # None => авто-выбор числовых
        self.sort_by_date = sort_by_date

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидаю pandas.DataFrame на входе.")
        df = X.copy()

        # авто-выбор колонок: все числовые, кроме date
        if self.columns is None:
            numeric_cols = df.select_dtypes(include=[np.number, "float", "int"]).columns.tolist()
            self.columns_ = [c for c in numeric_cols if c != self.date_col]
        else:
            self.columns_ = list(self.columns)

        # посчитаем пороги по каждому столбцу
        self.bounds_ = {}
        for col in self.columns_:
            s = pd.to_numeric(df[col], errors="coerce")
            q1 = np.nanpercentile(s, 25)
            q3 = np.nanpercentile(s, 75)
            iqr = q3 - q1
            lower = q1 - self.factor * iqr
            upper = q3 + self.factor * iqr
            self.bounds_[col] = (lower, upper)

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидаю pandas.DataFrame на входе.")
        if not hasattr(self, "bounds_"):
            raise RuntimeError("Сначала вызовите fit().")

        df = X.copy()

        # сортировка по дате (если нужно) — важна для "последнего корректного"
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        for col in self.columns_:
            s = pd.to_numeric(df[col], errors="coerce")

            lower, upper = self.bounds_[col]
            is_outlier = (s < lower) | (s > upper)

            # выбросы -> NaN, затем тянем последнее корректное значение вперёд
            s_masked = s.mask(is_outlier)
            s_filled = s_masked.ffill()

            # если выбросы в самом начале (нет "прошлого" значения) — оставляем исходные
            leading = s_filled.isna()
            if leading.any():
                s_filled[leading] = s[leading]

            df[col] = s_filled

        return df

In [8]:

pipe_filter = Pipeline([
    ("iqr_cf", IQRCarryForwardOutlierRemover(
        factor=1.5,
        date_col='date',
        columns=['close','open','high','low','volume','USD_RUB','close_RUR'],  # можно не указывать: выберет числовые сам
        sort_by_date=True
    ))
])

df_clean = pipe_filter.fit_transform(db)  # db — ваш исходный DataFrame

In [ ]:
df_clean.head() #type: ignore

,date,close,open,high,low,volume,USD_RUB,close_RUR
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70


## Создание признаков

In [10]:
# RSI

def calculate_rsi(new_data: pd.DataFrame, column='close', window=14)->pd.Series:
    delta = new_data[column].diff()
    gain = (delta.where(delta > 0, 0)).fillna(0)
    loss = (-delta.where(delta < 0, 0)).fillna(0)
    
    avg_gain = gain.rolling(window=window, min_periods=1).mean()
    avg_loss = loss.rolling(window=window, min_periods=1).mean()
    
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    rsi.fillna(0, inplace=True)
    return rsi

In [11]:
# Moving RSI
def calculate_moving_rsi(rsi: pd.Series, window=14):
    return rsi.rolling(window=window, min_periods=1).mean()

In [12]:
class TechIndicatorsTransformer(BaseEstimator, TransformerMixin):
    """Добавляет новые фичи:
    - RSI и его скользящее среднне
    - MACD, MACD_Signal, MACD_Hist
    - SMA короткое идлинное
    Args:
        BaseEstimator (_type_): _description_
        TransformerMixin (_type_): _description_
    """
    def __init__(
        self,
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
        date_col="date",
        sort_by_date=True,
        const_margin = 0.001  # ваш порог в единицах цены (или задайте свой)
    ):
        self.price_col = price_col
        self.rsi_window = rsi_window
        self.rsi_ma_window = rsi_ma_window
        self.sma_short = sma_short
        self.sma_long = sma_long
        self.date_col = date_col
        self.sort_by_date = sort_by_date
        self.const_margin = const_margin

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("TechnicalIndicatorsTransformer ожидает pandas.DataFrame")

        df = X.copy()

        # сортировка по дате (для временного ряда)
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        price = df[self.price_col].astype(float).values

        # RSI
        rsi = calculate_rsi(df, column=self.price_col, window=self.rsi_window)
        df[f"RSI_{self.rsi_window}"] = rsi
        df[f"RSI_{self.rsi_window}_MA{self.rsi_ma_window}"] = calculate_moving_rsi(
            rsi, window=self.rsi_ma_window
        )

        # MACD
        macd, macd_signal, macd_hist = talib.MACD(
            price, fastperiod=12, slowperiod=26, signalperiod=9
        )
        df["MACD"] = macd
        df["MACD_Signal"] = macd_signal
        df["MACD_Hist"] = macd_hist
        
        # Рассчитываем TripleEMA и MACD
        df['TEMA'] = talib.TEMA(df['close'], timeperiod=24)

        # SMA short/long
        df[f"SMA_{self.sma_short}"] = talib.SMA(price, timeperiod=self.sma_short)
        df[f"SMA_{self.sma_long}"] = talib.SMA(price, timeperiod=self.sma_long)

        df.fillna(0, inplace=True)
        
        return df

In [13]:
pipe_transform = Pipeline([
    ("ti", TechIndicatorsTransformer(
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
    )),
])

df_new = pipe_transform.fit_transform(df_clean)  # db = DataFrame с колонками ['date','close','open','high','low',...]
df_new.head()

,date,close,open,high,low,volume,USD_RUB,close_RUR,RSI_14,RSI_14_MA14,MACD,MACD_Signal,MACD_Hist,TEMA,SMA_20,SMA_50
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95,37.201007,9.300252,0.0,0.0,0.0,0.0,0.0,0.0
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70,56.369320,18.714065,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
class Trarget(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        price_col="close",
        date_col="date",
        const_margin = 0.001,  # ваш порог в единицах цены (или задайте свой)
        sort_by_date=True,
    ):
        self.price_col = price_col
        self.date_col = date_col
        self.const_margin = const_margin
        self.sort_by_date = sort_by_date
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("TechnicalIndicatorsTransformer ожидает pandas.DataFrame")

        df = X.copy()

        # сортировка по дате (для временного ряда)
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        price = df[self.price_col].astype(float).values
        
        # build_target
        nxt = df['close'].shift(-1)
        diff = nxt - df['close']
        y = np.where(diff >  self.const_margin,  1,
            np.where(diff < -self.const_margin, -1, 0))
        df['target'] = y
        
        return df


In [15]:
pipe_transform = Pipeline([
    ("ti", TechIndicatorsTransformer(
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
    )),
    ('target', Trarget(
        price_col='close',
        date_col='date',
        const_margin=0.02
    ))
])

df_new = pipe_transform.fit_transform(df_clean)  # db = DataFrame с колонками ['date','close','open','high','low',...]
df_new.head()

,date,close,open,high,low,volume,USD_RUB,close_RUR,RSI_14,RSI_14_MA14,MACD,MACD_Signal,MACD_Hist,TEMA,SMA_20,SMA_50,target
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,-1
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,-1
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95,37.201007,9.300252,0.0,0.0,0.0,0.0,0.0,0.0,1
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70,56.369320,18.714065,0.0,0.0,0.0,0.0,0.0,0.0,1


In [16]:
pipe_transform = Pipeline([
    ("iqr_cf", IQRCarryForwardOutlierRemover(
        factor=1.5,
        date_col='date',
        columns=['close','open','high','low','volume','USD_RUB','close_RUR'],  # можно не указывать: выберет числовые сам
        sort_by_date=True
    )),
    ("ti", TechIndicatorsTransformer(
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
    )),
    ('target', Trarget(
        price_col='close',
        date_col='date',
        const_margin=0.02
    ))
])

In [17]:
df_1 = pipe_transform.fit_transform(db)
df_1.head()

,date,close,open,high,low,volume,USD_RUB,close_RUR,RSI_14,RSI_14_MA14,MACD,MACD_Signal,MACD_Hist,TEMA,SMA_20,SMA_50,target
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,-1
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,-1
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95,37.201007,9.300252,0.0,0.0,0.0,0.0,0.0,0.0,1
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70,56.369320,18.714065,0.0,0.0,0.0,0.0,0.0,0.0,1


In [18]:
test = df_1.loc[df_1['date'].dt.year > 2023]
train = df_1.loc[df_1['date'].dt.year < 2024]

X_test = test[test.columns[:-1]]
y_test = test['target']

X_train = train[train.columns[:-1]]
y_train = train['target']

In [19]:
type(y_test)

pandas.core.series.Series

## Тестируем идеальную стратегию

### Backtesting

In [20]:
from backtesting import Backtest, Strategy

Loading BokehJS ...

In [21]:
# Backtesting.py ждёт колонки строго: Open, High, Low, Close, Volume
data = pd.DataFrame({
    'Open':   X_test['open'].astype(float),
    'High':   X_test['high'].astype(float),
    'Low':    X_test['low'].astype(float),
    'Close':  X_test['close'].astype(float),
    'Volume': X_test['volume'].fillna(0).astype(float),
})

In [22]:
# сигнал как Series по индексу цен
signals = y_test.astype(int).reindex(data.index).fillna(0)

In [23]:
# -------- Стратегия по готовому сигналу {-1,0,1} --------
class SignalLongShort(Strategy):
    signals: pd.Series = None
    frac = 0.99   # доля капитала на позицию (и для лонга, и для шорта)

    def init(self):
        s = self.signals.reindex(self.data.index).fillna(0).astype(int)
        self.sig = self.I(lambda: s.values)  # индикатор для доступа в next()

    def next(self):
        # проверка цены — если невалидна, ничего не делаем
        price = float(self.data.Close[-1])
        if not np.isfinite(price) or price <= 0:
            return

        sig  = int(self.sig[-1])
        prev = int(self.sig[-2]) if len(self.sig) > 1 else None
        if prev == sig:
            return  # ребалансируем только при смене сигнала

        # сначала закрываем что было
        if self.position:
            self.position.close()

        # затем открываем новую цель как долю капитала
        if sig == 1:
            # лонг на 99% капитала
            self.buy(size=self.frac)     # size<1 => доля капитала
        elif sig == -1:
            # шорт на 99% капитала
            self.sell(size=self.frac)    # size<1 => доля капитала
        # sig == 0 -> остаёмся в кэше (ничего не открываем)

In [24]:
# -------- Бенчмарк Buy & Hold --------
class BuyHold(Strategy):
    frac = 0.99
    def init(self):
        pass  # обязательный метод для Strategy
    
    def next(self):
        if not self.position:
            self.buy(size=self.frac)

In [25]:
# пробрасываем сигналы в стратегию
SignalLongShort.signals = signals

In [26]:
# -------- Создание бэктестов --------
bt_sig = Backtest(
    data, SignalLongShort,
    cash=100_000,            # начальный капитал
    commission=0.0005,       # 5 bps на сделку
    exclusive_orders=True,   # одна позиция за раз
    trade_on_close=False,    # сделки на следующей свече
    margin=1.0               # разрешает шорты (1х)
)

bt_bh = Backtest(
    data, BuyHold,
    cash=100_000,
    commission=0.0005,
    exclusive_orders=True,
    trade_on_close=False
)

In [27]:
stats_sig = bt_sig.run()
stats_bh  = bt_bh.run()

Backtest.run:   0%|          | 0/220 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/220 [00:00<?, ?bar/s]

In [ ]:
print("\t\t=== Signal strategy === Buy & Hold ===")
for i in ['Start','End']:
    print(f'{i:>18}:\t{stats_sig[i]}\t\t{stats_bh[i]}')
    
for i in ['Equity Final [$]','Return [%]','Max. Drawdown [%]']:
    print(f'{i:>18}:\t{np.round(stats_sig[i], 4):<10}\t{np.round(stats_bh[i], 4)}')

		=== Signal strategy === Buy & Hold ===
             Start:	3678.0		3678.0
               End:	3898.0		3898.0
  Equity Final [$]:	724887.1497	20825.173
        Return [%]:	624.8871  	-79.1748
 Max. Drawdown [%]:	-98.7634  	-79.2193


## Модель на базе тех анализа

In [29]:
def _ema(s: pd.Series, period: int) -> pd.Series:
    return pd.Series(s, index=s.index, dtype=float).ewm(span=period, adjust=False).mean()

def _tema(close: pd.Series, period: int) -> pd.Series:
    # TEMA = 3*EMA1 - 3*EMA2 + EMA3
    ema1 = _ema(close, period)
    ema2 = _ema(ema1, period)
    ema3 = _ema(ema2, period)
    return 3*ema1 - 3*ema2 + ema3

def _macd_from_close(close: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = _ema(close, fast)
    ema_slow = _ema(close, slow)
    macd = ema_fast - ema_slow
    macd_signal = _ema(macd, signal)
    return macd, macd_signal

class RuleSignalClassifier(BaseEstimator, ClassifierMixin):
    """
    Классификатор по правилу:
      buy  (1):  macd > macd_signal  AND  close > tema
      sell (-1): macd < macd_signal  AND  close < tema
      hold (0):  иначе

    Параметры:
      close_col: имя колонки цены закрытия
      macd_col, macd_signal_col, tema_col: имена колонок с индикаторами, если уже есть
      compute_if_missing: считать индикаторы из close, если колонок нет
      macd_fast, macd_slow, macd_signal: параметры MACD, если считаем
      tema_period: период TEMA, если считаем
      neutral_class: метка «нет сигнала» (по умолчанию 0)
    """

    
    def __init__(self,
                 close_col="close",
                 macd_col="macd",
                 macd_signal_col="macd_signal",
                 tema_col="tema",
                 compute_if_missing=True,
                 macd_fast=12, macd_slow=26, macd_signal=9,
                 tema_period=30,
                 neutral_class=0):
        self.close_col = close_col
        self.macd_col = macd_col
        self.macd_signal_col = macd_signal_col
        self.tema_col = tema_col
        self.compute_if_missing = compute_if_missing
        self.macd_fast = macd_fast
        self.macd_slow = macd_slow
        self.macd_signal = macd_signal
        self.tema_period = tema_period
        self.neutral_class = neutral_class

    # обучаться тут нечему — но sklearn ожидает fit
    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        # объявим порядок классов для predict_proba
        self.classes_ = np.array([-1, self.neutral_class, 1], dtype=int)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        df = self._ensure_indicators(X)
        macd = pd.to_numeric(df[self.macd_col], errors="coerce")
        macd_signal = pd.to_numeric(df[self.macd_signal_col], errors="coerce")
        close = pd.to_numeric(df[self.close_col], errors="coerce")
        tema = pd.to_numeric(df[self.tema_col], errors="coerce")

        # условия
        buy = (macd > macd_signal) & (close > tema)
        sell = (macd < macd_signal) & (close < tema)

        out = np.full(len(df), self.neutral_class, dtype=int)
        out[sell.fillna(False).values] = -1
        out[buy.fillna(False).values] = 1
        return out

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Жёсткие «вероятности»: 1.0 для предсказанного класса, 0.0 для остальных.
        Нужно лишь для совместимости со scorer’ами, если потребуется.
        """
        y = self.predict(X)
        proba = np.zeros((len(y), 3), dtype=float)  # порядок: [-1, neutral, +1]
        for i, label in enumerate(y):
            if label == -1:
                proba[i, 0] = 1.0
            elif label == 1:
                proba[i, 2] = 1.0
            else:
                proba[i, 1] = 1.0
        return proba

    # --- вспомогательное ---
    def _ensure_indicators(self, X: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        if self.close_col not in X.columns:
            raise KeyError(f"Не найдена колонка '{self.close_col}'")

        df = X.copy()

        # MACD
        need_macd = (self.macd_col not in df.columns) or (self.macd_signal_col not in df.columns)
        if need_macd:
            if not self.compute_if_missing:
                missing = [c for c in [self.macd_col, self.macd_signal_col] if c not in df.columns]
                raise KeyError(f"Отсутствуют {missing}, а вычислять запрещено (compute_if_missing=False)")
            macd, macd_sig = _macd_from_close(
                df[self.close_col].astype(float),
                fast=self.macd_fast, slow=self.macd_slow, signal=self.macd_signal
            )
            df[self.macd_col] = macd
            df[self.macd_signal_col] = macd_sig

        # TEMA
        if self.tema_col not in df.columns:
            if not self.compute_if_missing:
                raise KeyError(f"Отсутствует '{self.tema_col}', а вычислять запрещено (compute_if_missing=False)")
            df[self.tema_col] = _tema(df[self.close_col].astype(float), self.tema_period)

        return df

In [30]:
rule_clf = RuleSignalClassifier(
    close_col="close",
    macd_col="macd",
    macd_signal_col="macd_signal",
    tema_col="tema",
    compute_if_missing=True,  # посчитает индикаторы из close при отсутствии
    macd_fast=12, macd_slow=26, macd_signal=9,
    tema_period=30,
    neutral_class=0
)

In [31]:
rule_clf.fit(X_train)

RuleSignalClassifier()

In [32]:
y_pred = rule_clf.predict(X_test)

In [33]:
y_pred

array([ 0,  1,  1,  1,  1,  0,  1,  1,  1,  1,  1,  1,  0,  0,  1,  0,  0,
       -1, -1, -1,  0,  0, -1, -1, -1,  0, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,  0,  0,
        0, -1, -1, -1, -1, -1, -1, -1, -1,  0,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  0,  0,  0, -1,  1, -1, -1,  0,  1,  1,  1,  1,  0,
        0,  0, -1, -1, -1, -1, -1, -1,  1,  1,  0, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1,  0, -1, -1, -1,  0,  0, -1, -1,  1,  1,
        0,  1,  1,  1,  1,  1,  1,  1,  0,  1,  0, -1, -1, -1, -1, -1, -1,
       -1,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  0, -1, -1,  0, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0, -1, -1,  0, -1, -1,
       -1,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  0,  0,  0, -1, -1, -1, -1, -1, -1, -1,  0, -1, -1, -1,
        0,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0,  0,  1,  1,  1,  1])